# NISS Reconciliation: Original Outputs vs New Notebook Outputs

**Read-only.** This notebook only reads tables and files and shows results with `display()`. It does not write any table, file or temp view.

**What it compares**

| Check | Meaning |
|---|---|
| Row count | Old table vs new table |
| Schema | Columns only in one side, and data type differences |
| Row level | Rows in old but not in new, and rows in new but not in old (all columns compared as text, NULL equals NULL) |
| Totals | Sum of the key amount columns, overall and by state |
| Files | Transmittal, Summary, NISS layout file and Pre-final file, old folder vs new folder |

**Set the parameters, then run all cells.** The last cell shows a PASS / FAIL summary.

**Reading the result.** A few FAIL rows can be expected only where the original logic is not deterministic (for example `first()` after `groupBy`, or `row_number()` ordered by a constant). Investigate any FAIL in amounts, counts or row differences that are large.


In [ ]:
# PARAMETERS CELL
report_year = "2025"        # year of the new run
old_prefix = ""             # prefix for original tables, e.g. "DMAnalytics." (blank = default lakehouse)
new_prefix = ""             # prefix for new tables
sample_rows = "20"          # rows shown for each difference sample


In [ ]:
from functools import reduce
from pyspark.sql import DataFrame
from pyspark.sql import functions as F

YEAR = int(float(report_year))
SAMPLE = int(sample_rows)
RESULTS = []   # (check, object, status, detail), kept in memory only

# (original table, key amount columns, grouping column for the totals check or None)
TABLES = [
    ("OnSet_And_Refrence_Offset", [], None),
    ("Claim_Policy_NewApproachData", [], None),
    ("OnSet_And_Refrence_Offset_Address", [], None),
    ("Final_Transformed_Data",
     ["OnsetAmount", "OffsetAmount", "Final_Exposure", "PaidLossAmount", "ReserveAmount",
      "AllocatedLossAdjExp", "PaidCount", "ReserveCount"], "PrimaryRiskState"),
    ("NISS_Final_Report",
     ["WrittenExposure", "WrittenPremium", "PaidLosses", "PaidAllocatedLossAdjustmentExpenses",
      "OutstandingLossesIncludingALAE", "NumberOfPaidClaims", "NumberOfOutstandingClaims"], "StateCode"),
]

# (label, original folder, new folder, has header)
FILES = [
    ("TransmittalFILE", "Files/TransmittalFILE", f"Files/NISS_Reports/Yearly/{YEAR}/TransmittalFILE", False),
    ("SUMMARYFILE", "Files/SUMMARYFILE", f"Files/NISS_Reports/Yearly/{YEAR}/SUMMARYFILE", False),
    ("First_NISS_FORMAT_FILE", "Files/First_NISS_FORMAT_FILE", f"Files/NISS_Reports/Yearly/{YEAR}/First_NISS_FORMAT_FILE", False),
    ("Pre_Final_Report_File", "Files/Pre_Final_Report_File", f"Files/NISS_Reports/Yearly/{YEAR}/Pre_Final_Report_File", True),
]


def record(check, obj, ok, detail=""):
    """Store one result line in memory and print it."""
    RESULTS.append((check, obj, "PASS" if ok else "FAIL", str(detail)))
    print(f"{'PASS' if ok else 'FAIL'} | {check} | {obj} | {detail}")


def as_text(df: DataFrame, columns) -> DataFrame:
    """Select the given columns, all cast to string, in a fixed order."""
    return df.select(*[F.col(f"`{c}`").cast("string").alias(c) for c in sorted(columns)])


def compare_schema(old: DataFrame, new: DataFrame, name: str) -> list:
    """Report columns present on one side only and data type differences."""
    old_types, new_types = dict(old.dtypes), dict(new.dtypes)
    only_old = sorted(set(old_types) - set(new_types))
    only_new = sorted(set(new_types) - set(old_types))
    type_diff = sorted(c for c in set(old_types) & set(new_types) if old_types[c] != new_types[c])
    record("Schema", name, not (only_old or only_new or type_diff),
           f"only_old={only_old} only_new={only_new} type_diff={[(c, old_types[c], new_types[c]) for c in type_diff]}")
    return sorted(set(old_types) & set(new_types))


def compare_rows(old: DataFrame, new: DataFrame, columns, name: str) -> None:
    """Row level comparison on the common columns (counts and samples)."""
    old_text, new_text = as_text(old, columns), as_text(new, columns)
    old_only, new_only = old_text.exceptAll(new_text), new_text.exceptAll(old_text)
    old_only_count, new_only_count = old_only.count(), new_only.count()
    record("Rows only in old", name, old_only_count == 0, old_only_count)
    record("Rows only in new", name, new_only_count == 0, new_only_count)
    if old_only_count:
        print(f"Sample: rows only in OLD for {name}")
        display(old_only.limit(SAMPLE))
    if new_only_count:
        print(f"Sample: rows only in NEW for {name}")
        display(new_only.limit(SAMPLE))


def compare_totals(old: DataFrame, new: DataFrame, measures, group_col, name: str) -> None:
    """Compare sums of the key amount columns, overall and per group."""
    measures = [m for m in measures if m in old.columns and m in new.columns]
    if not measures:
        return
    def totals(df):
        aggs = [F.sum(F.col(f"`{m}`").cast("double")).alias(m) for m in measures]
        return df.groupBy(F.col(f"`{group_col}`").cast("string").alias("grp")).agg(*aggs)
    joined = totals(old).alias("o").join(totals(new).alias("n"), "grp", "full_outer")
    diffs = [F.coalesce(F.col(f"n.{m}"), F.lit(0.0)) - F.coalesce(F.col(f"o.{m}"), F.lit(0.0)) for m in measures]
    view = joined.select(
        "grp",
        *[F.col(f"o.{m}").alias(f"old_{m}") for m in measures],
        *[F.col(f"n.{m}").alias(f"new_{m}") for m in measures],
        *[d.alias(f"diff_{m}") for d, m in zip(diffs, measures)],
    )
    mismatch = view.filter(reduce(lambda a, b: a | b, [F.abs(F.col(f"diff_{m}")) > 0.005 for m in measures]))
    mismatch_count = mismatch.count()
    record("Totals by " + group_col, name, mismatch_count == 0, f"groups with differences={mismatch_count}")
    print(f"Totals by {group_col} for {name} (old, new, diff):")
    display(view.orderBy("grp"))


## Table reconciliation

In [ ]:
for base, measures, group_col in TABLES:
    old_name, new_name = f"{old_prefix}{base}", f"{new_prefix}{base}_Y{YEAR}"
    print("=" * 100)
    print(f"{old_name}  vs  {new_name}")
    try:
        old_df, new_df = spark.table(old_name), spark.table(new_name)
    except Exception as exc:
        record("Table exists", base, False, str(exc)[:200])
        continue
    old_count, new_count = old_df.count(), new_df.count()
    record("Row count", base, old_count == new_count, f"old={old_count} new={new_count}")
    common = compare_schema(old_df, new_df, base)
    compare_rows(old_df, new_df, common, base)
    if measures:
        compare_totals(old_df, new_df, measures, group_col, base)


## File reconciliation

In [ ]:
for label, old_path, new_path, has_header in FILES:
    print("=" * 100)
    print(f"{old_path}  vs  {new_path}")
    try:
        reader = spark.read.option("header", "true" if has_header else "false")
        old_df, new_df = reader.csv(old_path), reader.csv(new_path)
        old_count, new_count = old_df.count(), new_df.count()
    except Exception as exc:
        record("File readable", label, False, str(exc)[:200])
        continue
    record("Row count", label, old_count == new_count, f"old={old_count} new={new_count}")
    common = sorted(set(old_df.columns) & set(new_df.columns))
    if has_header:
        compare_schema(old_df, new_df, label)
    compare_rows(old_df, new_df, common, label)


## Summary

In [ ]:
summary = spark.createDataFrame(RESULTS, "Check STRING, Object STRING, Status STRING, Detail STRING")
failed = [r for r in RESULTS if r[2] == "FAIL"]
print(f"Checks run: {len(RESULTS)}   PASS: {len(RESULTS) - len(failed)}   FAIL: {len(failed)}")
display(summary.orderBy(F.when(F.col("Status") == "FAIL", 0).otherwise(1), "Object", "Check"))
